In [0]:
dbutils.widgets.removeAll()
dbutils.widgets.text("fecha_procesado","","fecha_procesado")
dbutils.widgets.text("id_proceso","","id_proceso")

id_proceso = dbutils.widgets.get("id_proceso")
fecha_procesado = dbutils.widgets.get("fecha_procesado")

In [0]:
%sql
CREATE OR REPLACE TABLE workspace.silver.inventario_ventas_tiendas AS
SELECT
  i.id_producto,
  i.stock,
  i.fecha_actualizacion,
  v.fecha,
  v.cantidad,
  t.id_tienda,
  t.ciudad,
  current_timestamp() as fecha_procesado
FROM workspace.bronze.inventario i
INNER JOIN workspace.bronze.ventas_diarias_2m v
  ON i.id_producto = v.id_producto
INNER JOIN workspace.bronze.tiendas t
  ON v.id_tienda = t.id_tienda;

In [0]:
df = spark.sql("SELECT COUNT(*) AS rows FROM workspace.silver.inventario_ventas_tiendas WHERE DATE(fecha_procesado) = DATE(current_timestamp())")

for i in df.collect():
    if i['rows'] > 0:
        dbutils.notebook.exit(1)
    else:
        dbutils.notebook.exit(0)
